<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230824_ar2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler


In [23]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')


In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(s30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 36

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)




Epoch 1/50
157/157 [==============================] - 30s 116ms/step - loss: 0.0013 - mae: 0.0241 - val_loss: 2.9406e-04 - val_mae: 0.0137
Epoch 2/50
157/157 [==============================] - 18s 116ms/step - loss: 3.6520e-04 - mae: 0.0127 - val_loss: 3.5477e-04 - val_mae: 0.0149
Epoch 3/50
157/157 [==============================] - 22s 140ms/step - loss: 2.7689e-04 - mae: 0.0109 - val_loss: 2.9910e-04 - val_mae: 0.0136
Epoch 4/50
157/157 [==============================] - 21s 132ms/step - loss: 2.1996e-04 - mae: 0.0097 - val_loss: 2.5965e-04 - val_mae: 0.0127
Epoch 5/50
157/157 [==============================] - 18s 115ms/step - loss: 2.4116e-04 - mae: 0.0094 - val_loss: 2.2446e-04 - val_mae: 0.0117
Epoch 6/50
157/157 [==============================] - 19s 123ms/step - loss: 2.0236e-04 - mae: 0.0087 - val_loss: 2.8098e-04 - val_mae: 0.0131
Epoch 7/50
157/157 [==============================] - 18s 114ms/step - loss: 1.6113e-04 - mae: 0.0082 - val_loss: 2.6207e-04 - val_mae: 0.0127
Epo

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(s30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 36

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)




In [ ]:
predictions_30s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_40s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_50s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_70s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_100s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100s2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.concat([predictions_30c,predictions_30c2],axis=1)
c40 = pd.concat([predictions_40c,predictions_40c2],axis=1)
c50 = pd.concat([predictions_50c,predictions_50c2],axis=1)
c70 = pd.concat([predictions_70c,predictions_70c2],axis=1)
c100 = pd.concat([predictions_100c,predictions_100c2],axis=1)

s30 = pd.concat([predictions_30s,predictions_30s2],axis=1)
s40 = pd.concat([predictions_40s,predictions_40s2],axis=1)
s50 = pd.concat([predictions_50s,predictions_50s2],axis=1)
s70 = pd.concat([predictions_70s,predictions_70s2],axis=1)
s100 = pd.concat([predictions_100s,predictions_100s2],axis=1)

In [ ]:
c30  = pd.DataFrame(c30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c40  = pd.DataFrame(c40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c50  = pd.DataFrame(c50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c70  = pd.DataFrame(c70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c100  = pd.DataFrame(c100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s30  = pd.DataFrame(s30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s40  = pd.DataFrame(s40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s50  = pd.DataFrame(s50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s70  = pd.DataFrame(s70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
s100  = pd.DataFrame(s100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])

In [ ]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

s30.columns = ['YL_M1_B1_W1_s30', 'YR_M1_B1_W1_s30', 'YL_M1_B1_W2_s30', 'YR_M1_B1_W2_s30']
s40.columns = ['YL_M1_B1_W1_s40', 'YR_M1_B1_W1_s40', 'YL_M1_B1_W2_s40', 'YR_M1_B1_W2_s40']
s50.columns = ['YL_M1_B1_W1_s50', 'YR_M1_B1_W1_s50', 'YL_M1_B1_W2_s50', 'YR_M1_B1_W2_s50']
s70.columns = ['YL_M1_B1_W1_s70', 'YR_M1_B1_W1_s70', 'YL_M1_B1_W2_s70', 'YR_M1_B1_W2_s70']
s100.columns = ['YL_M1_B1_W1_s100', 'YR_M1_B1_W1_s100', 'YL_M1_B1_W2_s100', 'YR_M1_B1_W2_s100']

In [ ]:
c30

array([[ 0.02126211, -0.00605943,  0.03612377, -0.02745003],
       [ 0.01889343, -0.00298353,  0.04117558, -0.03263278],
       [ 0.01125363,  0.00320626,  0.04452606, -0.03732274],
       ...,
       [ 0.00376992,  0.00257052,  0.01425002, -0.00873982],
       [ 0.00652188,  0.00012569,  0.00467195,  0.00084792],
       [ 0.00862526, -0.00194681, -0.0002387 ,  0.0057446 ]],
      dtype=float32)

In [ ]:
answer = pd.concat([answer_sample.Distance, s30,s40,s50,s70,s100,c30,c40,c50,c70,c100], axis=1)

In [ ]:
answer

,Distance,YL_M1_B1_W1_s30,YR_M1_B1_W1_s30,YL_M1_B1_W2_s30,YR_M1_B1_W2_s30,YL_M1_B1_W1_s40,YR_M1_B1_W1_s40,YL_M1_B1_W2_s40,YR_M1_B1_W2_s40,YL_M1_B1_W1_s50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.021262,-0.006059,0.036124,-0.027450,0.002733,0.006266,0.001176,0.005620,0.008963,...,0.016798,-0.007281,0.008570,0.002362,0.013847,-0.005630,0.008920,0.000447,0.010149,-0.003260
1,2500.50,0.018893,-0.002984,0.041176,-0.032633,-0.002467,0.012125,0.006006,0.000805,0.002908,...,0.024805,-0.014375,0.001207,0.008957,0.021211,-0.012390,0.003529,0.005665,0.019009,-0.011519
2,2500.75,0.011254,0.003206,0.044526,-0.037323,-0.012184,0.020639,0.009545,-0.003153,-0.010249,...,0.033419,-0.023192,-0.011523,0.018417,0.027361,-0.018821,-0.009655,0.015675,0.030366,-0.022466
3,2501.00,0.004753,0.007461,0.044155,-0.038052,-0.021290,0.028001,0.009562,-0.003468,-0.024616,...,0.038600,-0.028774,-0.024799,0.027692,0.030751,-0.022912,-0.022894,0.025024,0.038446,-0.030673
4,2501.25,0.002106,0.008624,0.040752,-0.035146,-0.023518,0.030085,0.008626,-0.002483,-0.030269,...,0.042978,-0.033480,-0.030969,0.032028,0.034401,-0.026873,-0.026846,0.027969,0.043958,-0.036235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,0.007718,-0.000202,0.032223,-0.025558,0.015068,-0.006111,0.039828,-0.030747,0.015677,...,0.053031,-0.042401,0.013386,-0.004949,0.053090,-0.042846,0.009380,-0.004479,0.052779,-0.043282
1995,2999.00,0.006235,0.000625,0.025453,-0.019328,0.008958,-0.000666,0.034552,-0.026492,0.008961,...,0.049649,-0.039769,0.008478,-0.000738,0.047665,-0.038120,0.003734,0.000247,0.044987,-0.036594
1996,2999.25,0.003770,0.002571,0.014250,-0.008740,0.003005,0.004768,0.023047,-0.015946,0.002874,...,0.042089,-0.033556,-0.000424,0.005708,0.035017,-0.027068,-0.002712,0.005407,0.033181,-0.026081
1997,2999.50,0.006522,0.000126,0.004672,0.000848,0.006862,0.000718,0.010482,-0.003666,0.005750,...,0.030875,-0.023369,0.002062,0.002807,0.018990,-0.012627,-0.000287,0.003282,0.019535,-0.013433


In [ ]:
answer.to_csv('/content/drive/My Drive/철도/answer20230824_ar.csv', index=False)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 예측과 실제값 간의 MAE, MSE, RMSE, R-squared 계산
mae = mean_absolute_error(y_test_s, predictions_s)
mse = mean_squared_error(y_test_s, predictions_s)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_s, predictions_s)

# 결과 출력
print("Evaluation metrics for data_s:")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)


Evaluation metrics for data_s:
MAE: 0.004689485690516887
MSE: 5.018317484975397e-05
RMSE: 0.007084008388599915
R-squared: 0.9032700239745032
